# Lesson 3: Who speaks next

For each cell: **guess what it will print first**, then press ▶ and check.

From [Build an AI Team](https://gradlabs.in/learn/agent-team/lessons/lesson3.html).

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The shared chat, and a list of who the LLM wrote for.
# Every LLM write costs time and money, so we count them: len(calls).
chat = []
calls = []

def post(who, text):
    chat.append({"who": who, "text": text})
    print(f"{who}: {text}")

BOTS = ["Marketing", "Production", "Operations"]
INSTRUCTIONS = {
    "Marketing": "Plan Instagram posts and captions for Amal Designers. Use only facts in the chat.",
    "Production": "Check sizes, stock and prices. Catalogue: pastel party frock, sizes 1 to 8 years, ₹1,450.",
    "Operations": "Check packing and shipping. Shipping takes 3 days to most of India.",
}

def text_for(name):
    lines = [INSTRUCTIONS[name], "", "Chat so far:"]
    for m in chat:
        lines.append(m["who"] + ": " + m["text"])
    lines.append(name + ":")
    return "\n".join(lines)

In [ ]:
# Fake LLM replies for each bot, in order. A fresh copy for each run.
def fresh_replies():
    return {
        "Marketing": ["Caption: “Twirl-ready pastel frock 💫 Sizes 1 to 12 years. Ships in 2 days!” @Production please check.",
                      "Fixed: “…Sizes 1 to 8 years. Ships in 3 days.” @CEO ready to post."],
        "Production": ["Sizes are 1 to 8 years, not 12. @Operations please check the shipping."],
        "Operations": ["Shipping takes 3 days, not 2. @Marketing"],
    }

def fake_llm(name, text):
    calls.append(name)
    return replies[name].pop(0)

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Way 1: @mentions. The bots named with @ reply next. No LLM write to decide.
def by_mentions(text):
    names = []
    for name in BOTS:
        if "@" + name in text:
            names.append(name)
    return names

print(by_mentions("@Production please check."))

In [ ]:
# Way 2: a coordinator reads the latest message and names who replies next.
# It is an LLM too, so it costs one write each time.
def by_coordinator(text):
    calls.append("Coordinator")
    name = coordinator_replies.pop(0)
    if name in BOTS:
        return [name]
    return []   # not a bot's name: wait for the CEO

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The loop: after each message, ask the router who replies next.
# chat[-1] is the last message (-1 counts from the end).
def run_chat(task, router, round_limit=5):
    post("CEO", task)
    for i in range(round_limit):
        names = router(chat[-1]["text"])
        if names == []:
            return "Waiting for the CEO."
        for name in names:
            post(name, fake_llm(name, text_for(name)))
    return "Stopped: the round limit was reached."

replies = fresh_replies()
print(run_chat("Plan a post for the pastel frock. @Marketing", by_mentions))
print("LLM writes:", len(calls))

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The same job with a coordinator: start a fresh chat and count again
chat.clear()
calls.clear()
replies = fresh_replies()
coordinator_replies = ["Marketing", "Production", "Operations", "Marketing", "CEO"]
print(run_chat("Plan a post for the pastel frock. @Marketing", by_coordinator))
print("LLM writes:", len(calls))

## Now a real LLM

Now with a small free model (Qwen2.5 1.5B) writing the replies.
The first run downloads it (about 3 GB), so give it a few minutes.

Small models make mistakes. The program checks everything they write, and its limits stop a chat that goes wrong.

In [ ]:
# A real LLM: a small, free model that runs inside Colab
# (faster with a GPU: Runtime → Change runtime type → T4 GPU)
import torch                        # the library that does the model's math
from transformers import pipeline   # a ready-made way to load a model and use it

In [ ]:
# Load Qwen2.5 1.5B (about 3 GB, downloaded the first time)
model = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                 device=0 if torch.cuda.is_available() else -1)

def ask_model(text, max_new_tokens=80):
    out = model([{"role": "user", "content": text}], max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]   # the model's reply: the last message

NAMES = ["CEO", "Coordinator"] + BOTS

In [ ]:
# A bot's reply, cut where another name starts (as in Lesson 2)
def real_llm(name, text):
    calls.append(name)
    kept = []
    for line in ask_model(text).strip().split("\n"):
        if line.split(":")[0] in NAMES:
            if kept:
                break
            line = line.split(":", 1)[1]
        kept.append(line.strip())
    return " ".join(kept).strip()

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The coordinator: its own instructions, the chat, then "Next:"
COORDINATOR = """You run the team chat at Amal Designers, a kids' dress brand.
Bots: Marketing (writes captions), Production (checks sizes and prices), Operations (checks shipping).
Read the chat. Who should reply next? Reply with one name only.
Reply CEO when the caption is ready, or when the CEO must decide."""

def real_coordinator(text):
    calls.append("Coordinator")
    lines = [COORDINATOR, "", "Chat so far:"]
    for m in chat:
        lines.append(m["who"] + ": " + m["text"])
    lines.append("Next:")
    answer = ask_model("\n".join(lines), max_new_tokens=10)
    print("   (coordinator:", answer.strip() + ")")
    for name in BOTS:
        if name in answer:   # the first bot name in its answer
            return [name]
    return []                # no bot named: wait for the CEO

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# run_chat calls fake_llm, so point that name at the real LLM
fake_llm = real_llm
chat.clear()
calls.clear()
print(run_chat("Plan an Instagram post for the new pastel party frock.", real_coordinator))
print("LLM writes:", len(calls))